<a href="https://colab.research.google.com/github/dilshodbekio/titanic_classification/blob/main/titanic_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

df = pd.read_csv("train.csv")

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
df.isna().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [ ]:
df["Survived"].value_counts()

,count
Survived,
0,549
1,342


In [ ]:
df["Survived"].value_counts(normalize=True) * 100

,proportion
Survived,
0,61.616162
1,38.383838


In [ ]:
from sklearn.model_selection import train_test_split

#Targetni featurelardan ajratamiz
X = df.drop(columns=["Survived"])
y = df["Survived"]

# 70% train, 30% vaqtinchalik qism
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)


#Qolgan 30% ni teng ikkiga: 15% validation, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

In [ ]:
for name, target in [
    ("Train", y_train),
    ("Validation", y_val),
    ("Test", y_test)
]:
  print(f"\n{name}: {len(target)} ta qator")
  print(target.value_counts(normalize=True).mul(100).round(2))


Train: 623 ta qator
Survived
0    61.64
1    38.36
Name: proportion, dtype: float64

Validation: 134 ta qator
Survived
0    61.19
1    38.81
Name: proportion, dtype: float64

Test: 134 ta qator
Survived
0    61.94
1    38.06
Name: proportion, dtype: float64


In [ ]:
#To'ldirish uchun qiymatlarni faqat traindan olamiz
age_median = X_train["Age"].median()
embarked_mode = X_train["Embarked"].mode().iloc[0]

print("Age medianasi:", age_median)
print("Eng ko'p uchragan port:", embarked_mode)

Age medianasi: 29.0
Eng ko'p uchragan port: S


In [ ]:
#Har bir qisiming alohida nusxasini olamiz
X_train = X_train.copy()
X_val = X_val.copy()
X_test = X_test.copy()


for data in [X_train, X_val, X_test]:
  data["Age"] = data["Age"].fillna(age_median)
  data["Embarked"] = data["Embarked"].fillna(embarked_mode)
  data.drop(columns=["Cabin"], inplace=True)

In [ ]:
for name, data in [
    ("Train", X_train),
    ("Validation", X_val),
    ("Test", X_test)
]:
  print(f"\n{name} - bo'sh qiymatlar:")
  print(data.isna().sum())


Train - bo'sh qiymatlar:
PassengerId    0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64

Validation - bo'sh qiymatlar:
PassengerId    0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64

Test - bo'sh qiymatlar:
PassengerId    0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64


In [ ]:
drop_columns = ["PassengerId", "Name", "Ticket"]

X_train = X_train.drop(columns=drop_columns)
X_val = X_val.drop(columns=drop_columns)
X_test = X_test.drop(columns=drop_columns)

In [ ]:
from sklearn.preprocessing import OneHotEncoder

categorical_columns = ["Sex", "Embarked"]

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

train_encoded = encoder.fit_transform(X_train[categorical_columns])
val_encoded = encoder.transform(X_val[categorical_columns])
test_encoded = encoder.transform(X_test[categorical_columns])

In [ ]:
encoded_columns = encoder.get_feature_names_out(categorical_columns)

train_encoded_df = pd.DataFrame(
    train_encoded,
    columns=encoded_columns,
    index=X_train.index
)

val_encoded_df = pd.DataFrame(
    val_encoded,
    columns=encoded_columns,
    index=X_val.index
)

test_encoded_df = pd.DataFrame(
    test_encoded,
    columns=encoded_columns,
    index=X_test.index
)

In [ ]:
X_train = pd.concat(
    [X_train.drop(columns=categorical_columns), train_encoded_df],
    axis=1
)

X_val = pd.concat(
    [X_val.drop(columns=categorical_columns), val_encoded_df],
    axis=1
)


In [ ]:
X_test = pd.concat(
    [X_test.drop(columns=categorical_columns), test_encoded_df],
    axis=1
)

In [ ]:
from sklearn.preprocessing import StandardScaler

numeric_columns = ["Pclass", "Age", "SibSp", "Parch", "Fare"]

scaler = StandardScaler()

X_train[numeric_columns] = scaler.fit_transform(
    X_train[numeric_columns]
)

X_val[numeric_columns] = scaler.transform(
    X_val[numeric_columns]
)

X_test[numeric_columns] = scaler.transform(
    X_test[numeric_columns]
)

In [ ]:
X_train.head()

,Pclass,Age,SibSp,Parch,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
748,-1.589202,-0.831890,0.562957,-0.448665,0.465738,0.0,1.0,0.0,0.0,1.0
45,0.819734,-0.064873,-0.474682,-0.448665,-0.478269,0.0,1.0,0.0,0.0,1.0
28,0.819734,-0.064873,-0.474682,-0.448665,-0.481848,1.0,0.0,0.0,1.0,0.0
633,-1.589202,-0.064873,-0.474682,-0.448665,-0.646954,0.0,1.0,0.0,0.0,1.0
403,0.819734,-0.141574,0.562957,-0.448665,-0.314823,0.0,1.0,0.0,0.0,1.0


In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [ ]:
y_val_pred = model.predict(X_val)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("Accuracy:", accuracy_score(y_val, y_val_pred))
print("Precision:", precision_score(y_val, y_val_pred, zero_division=0))
print("Recall:", recall_score(y_val, y_val_pred, zero_division=0))
print("F1:", f1_score(y_val, y_val_pred, zero_division=0))

print("\nConfusion matrix:")
print(confusion_matrix(y_val, y_val_pred))

Accuracy: 0.8582089552238806
Precision: 0.851063829787234
Recall: 0.7692307692307693
F1: 0.8080808080808081

Confusion matrix:
[[75  7]
 [12 40]]


In [ ]:
results = df.loc[X_val.index].copy()

results["Prediction"] = y_val_pred

errors = results[
    results["Survived"] != results["Prediction"]
]

errors[
    ["Pclass", "Sex", "Age", "Fare", "Survived", "Prediction"]
].head(10)

,Pclass,Sex,Age,Fare,Survived,Prediction
377,1,male,27.0,211.5000,0,1
578,3,female,NaN,14.4583,0,1
502,3,female,NaN,7.6292,0,1
165,3,male,9.0,20.5250,1,0
279,3,female,35.0,20.2500,1,0
643,3,male,NaN,56.4958,1,0
423,3,female,28.0,14.4000,0,1
570,2,male,62.0,10.5000,1,0
549,2,male,8.0,36.7500,1,0
17,2,male,NaN,13.0000,1,0


In [ ]:
results["Survival_probability"] = model.predict_proba(X_val)[:, 1]

errors = results[
    results["Survived"] != results["Prediction"]
]

errors[
    ["Sex", "Pclass", "Age", "Survived", "Prediction", "Survival_probability"]
].head(10)

,Sex,Pclass,Age,Survived,Prediction,Survival_probability
377,male,1,27.0,0,1,0.661138
578,female,3,NaN,0,1,0.590298
502,female,3,NaN,0,1,0.718769
165,male,3,9.0,1,0,0.180937
279,female,3,35.0,1,0,0.440451
643,male,3,NaN,1,0,0.104126
423,female,3,28.0,0,1,0.510750
570,male,2,62.0,1,0,0.071244
549,male,2,8.0,1,0,0.357565
17,male,2,NaN,1,0,0.237729


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score


baseline = DummyClassifier(strategy="most_frequent")

baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_val)

print("Baseline accuracy:", accuracy_score(y_val, baseline_pred))
print("Baseline F1:", f1_score(y_val, baseline_pred, zero_division=0))

Baseline accuracy: 0.6119402985074627
Baseline F1: 0.0


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

numeric_columns = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical_columns = ["Sex", "Embarked"]

#Oldingi train va validation ga tegishli xom qatorlar
dev_index = X_train.index.append(X_val.index)


X_dev = df.loc[dev_index, numeric_columns + categorical_columns].copy()
y_dev = df.loc[dev_index, "Survived"].copy()

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns)
])

cv_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_validate(
    cv_pipeline,
    X_dev,
    y_dev,
    cv=cv,
    scoring=["accuracy", "f1"]
)

print("Har fold accuracy:", scores["test_accuracy"].round(3))
print("O'rtacha accuracy:", scores["test_accuracy"].mean())
print("Accuracy std:", scores["test_accuracy"].std())

print("\nHar fold F1:", scores["test_f1"].round(3))
print("O'rtacha F1:", scores["test_f1"].mean())

Har fold accuracy: [0.77  0.822 0.801 0.815 0.808]
O'rtacha accuracy: 0.8031892645521086
Accuracy std: 0.018124727692493982

Har fold F1: [0.69  0.752 0.737 0.745 0.734]
O'rtacha F1: 0.7317601339106918


In [ ]:
from sklearn.base import clone

# Xuddi shu pipeline'ning yangi nusxasi
final_model = clone(cv_pipeline)

# Train + validation ning barcha xom ma'lumotlarida o'qitamiz
final_model.fit(X_dev, y_dev)

# Yakuniy test'ning xom ma'lumotlari
X_test_raw = df.loc[X_test.index, X_dev.columns]

final_pred = final_model.predict(X_test_raw)

print("Test accuracy:", accuracy_score(y_test, final_pred))
print("Test precision:", precision_score(y_test, final_pred, zero_division=0))
print("Test recall:", recall_score(y_test, final_pred, zero_division=0))

Test accuracy: 0.7388059701492538
Test precision: 0.6481481481481481
Test recall: 0.6862745098039216
